# Clean CSV with Pandas
**Author:** Wajid Ali Saleem Chaudhry  
**Project:** https://roadmap.sh/projects/clean-csv  
**Dataset:** [Messy Employee Dataset](https://www.kaggle.com/datasets/desolution01/messy-employee-dataset)
(`data/Messy_Employee_dataset.csv`) — a synthetic Kaggle dataset with
missing values, a combined department/region field, mixed dtypes, and
categorical values that need standardizing.

## C1 — Setup & Imports

In [1]:
# Author:      Wajid Ali Saleem Chaudhry
# Description: Imports for the Clean CSV notebook.

import pandas as pd

## C2 — Load & Inspect

In [2]:
# --- Load Raw Dataset ---
df = pd.read_csv("data/Messy_Employee_dataset.csv")
print(df.shape)
df.info()

(1020, 12)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 88.8 KB


In [3]:
# --- Preview & Missing-Value Audit ---
# Age/Salary have gaps; Department_Region packs two fields into one;
# Performance_Score/Remote_Work read in as generic object dtype even
# though their value sets are small and fixed.
df.head(10)

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False
5,EMP1005,Alice,Garcia,40.0,Sales-Texas,Inactive,6/10/2020,88642.84,alice.garcia@example.com,-5409003485,Good,False
6,EMP1006,Frank,Jones,NaN,Admin-Nevada,Active,4/3/2020,96288.43,frank.jones@example.com,-4518376063,Good,False
7,EMP1007,Bob,Jones,30.0,Cloud Tech-Florida,Inactive,7/17/2022,94497.91,bob.jones@example.com,-4134327559,Average,True
8,EMP1008,Frank,Davis,35.0,Admin-Nevada,Inactive,12/8/2023,115565.82,frank.davis@example.com,-4177656123,Excellent,True
9,EMP1009,Charlie,Johnson,NaN,DevOps-New York,Active,8/4/2022,76561.88,charlie.johnson@example.com,-8156985699,Excellent,True


In [4]:
# --- Nulls & Duplicates ---
print(df.isnull().sum())
print("duplicate rows:", df.duplicated().sum())

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64
duplicate rows: 0


## C3 — Handle Technical Issues (Encoding & Structure)

In [5]:
# --- Confirm Encoding & Delimiter ---
# Raw bytes are checked before trusting read_csv's defaults: any byte
# above 127 would signal non-UTF-8 encoding, and this file has none,
# so encoding="utf-8" and the default comma delimiter are both safe.
with open("data/Messy_Employee_dataset.csv", "rb") as f:
    raw = f.read()
print("non-ascii bytes:", sum(b > 127 for b in raw))
print("first line:", raw.split(b"\n")[0])

non-ascii bytes: 0
first line: b'Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work\r'


In [6]:
# --- Split Department_Region ---
# "DevOps-California" packs a department and a US state into one
# column separated by a single hyphen, so a one-split on the first
# "-" cleanly recovers both fields.
dept_region = df["Department_Region"].str.split("-", n=1, expand=True)
df["Department"] = dept_region[0]
df["Region"] = dept_region[1]
df = df.drop(columns=["Department_Region"])

print(sorted(df["Department"].unique()))
print(sorted(df["Region"].unique()))

['Admin', 'Cloud Tech', 'DevOps', 'Finance', 'HR', 'Sales']
['California', 'Florida', 'Illinois', 'Nevada', 'New York', 'Texas']


## C4 — Fix Data Types

In [7]:
# --- Cast Numeric Columns ---
# Age uses pandas' nullable "Int64" (capital I) instead of the numpy
# "int64" so the existing NaNs survive the cast instead of forcing a
# float dtype or raising. Salary is plain float64. Remote_Work's
# "TRUE"/"FALSE" values are already inferred as bool by read_csv, so
# no explicit cast is needed there.
df["Age"] = df["Age"].astype("Int64")
df["Salary"] = df["Salary"].astype("float64")

df[["Age", "Salary", "Remote_Work"]].dtypes

Age              Int64
Salary         float64
Remote_Work       bool
dtype: object

In [8]:
# --- Cast Performance_Score as an Ordered Category ---
# The four values have a natural ranking, so an ordered category lets
# later analysis compare/sort performance without a manual mapping.
score_order = ["Poor", "Average", "Good", "Excellent"]
df["Performance_Score"] = pd.Categorical(
    df["Performance_Score"], categories=score_order, ordered=True
)

df["Performance_Score"].dtype

CategoricalDtype(categories=['Poor', 'Average', 'Good', 'Excellent'], ordered=True, categories_dtype=str)

## C5 — Convert Dates

In [9]:
# --- Parse Join_Date ---
# Values are consistently "M/D/YYYY", but errors="coerce" is kept so
# any future row with an unparseable date becomes NaT instead of
# raising and silently corrupting the whole column.
df["Join_Date"] = pd.to_datetime(
    df["Join_Date"], format="%m/%d/%Y", errors="coerce"
)
print("unparseable dates:", df["Join_Date"].isna().sum())
df["Join_Date"].dtype

unparseable dates: 0


dtype('<M8[us]')

## C6 — Standardize Categories

In [10]:
# --- Strip Whitespace & Check Capitalization ---
# Values are stripped on principle, then checked for case
# inconsistency by comparing each column's unique values against
# their casefolded form. Blind .str.title() is avoided here because
# it would mangle correctly-cased abbreviations like "DevOps"/"HR"
# into "Devops"/"Hr" — this dataset turns out to already be
# consistently capitalized, so no casing rewrite is applied.
for col in ["Status", "Department", "Region"]:
    df[col] = df[col].str.strip()

for col in ["Status", "Department", "Region"]:
    values = df[col].dropna().unique()
    casefolded = {v.casefold(): v for v in values}
    print(col, "case-consistent:", len(casefolded) == len(values))

print(sorted(df["Status"].unique()))
print(sorted(df["Department"].unique()))
print(sorted(df["Region"].unique()))

Status case-consistent: True
Department case-consistent: True
Region case-consistent: True
['Active', 'Inactive', 'Pending']
['Admin', 'Cloud Tech', 'DevOps', 'Finance', 'HR', 'Sales']
['California', 'Florida', 'Illinois', 'Nevada', 'New York', 'Texas']


## C7 — Export & Audit

In [11]:
# --- Final Dtype & Null Audit ---
print(df.dtypes)
print(df.isnull().sum())

Employee_ID                     str
First_Name                      str
Last_Name                       str
Age                           Int64
Status                          str
Join_Date            datetime64[us]
Salary                      float64
Email                           str
Phone                         int64
Performance_Score          category
Remote_Work                    bool
Department                      str
Region                          str
dtype: object
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
Department             0
Region                 0
dtype: int64


In [12]:
# --- Export Cleaned Dataset ---
df.to_csv("data/employees_cleaned.csv", index=False)
print(f"Exported {len(df)} rows, {len(df.columns)} columns.")

Exported 1020 rows, 13 columns.
